In [ ]:
libraries<-c("WGCNA","ggplot2", "reshape2", "ComplexHeatmap","pheatmap","dplyr","textshape","tidyr")
suppressPackageStartupMessages(lapply(libraries, require, character.only = TRUE))

In [ ]:
#Load count data TMM normalized 
tmm_norm<- read.delim("your_path/normCounts_tmm.tsv",header = T)

In [13]:
tmm_norm <- rownames_to_column(tmm_norm, var = "GeneID")

In [14]:
colnames(tmm_norm)

[1] "GeneID"     "C195_60H_1" "C195_60H_2" "C195_60H_3" "C195_60H_4"
 [6] "C195_84H_1" "C195_84H_2" "C195_84H_3" "C195_84H_4" "C195_24H_1"
[11] "C195_24H_2" "C195_24H_3" "C195_24H_4" "C195_C1"    "C195_C2"   
[16] "C195_C3"    "C195_C4"    "C98_60H_1"  "C98_60H_2"  "C98_60H_3" 
[21] "C98_60H_4"  "C98_84H_1"  "C98_84H_2"  "C98_84H_3"  "C98_84H_4" 
[26] "C98_24H_1"  "C98_24H_2"  "C98_24H_3"  "C98_24H_4"  "C98_C1"    
[31] "C98_C2"     "C98_C3"     "C98_C4"

In [15]:
tmm_norm <- tmm_norm %>% select(GeneID, C98_C1, C98_C2, C98_C3, C98_C4, C98_24H_1, C98_24H_2, C98_24H_3, C98_24H_4, C98_60H_1, C98_60H_2, C98_60H_3, C98_60H_4, C98_84H_1, C98_84H_2, C98_84H_3, C98_84H_4, C195_C1, C195_C2, C195_C3, C195_C4, C195_24H_1, C195_24H_2, C195_24H_3, C195_24H_4, C195_60H_1, C195_60H_2, C195_60H_3, C195_60H_4, C195_84H_1, C195_84H_2, C195_84H_3, C195_84H_4)

In [ ]:
#Metadata
#sampledatainfo.txt file was a table with metadata info, similar to: 
#sample	hpi	genotype	biosample	response	tissue	treatment
#C195_60H_1	060HPI	C195	R1	Susceptible	hypocotyl	C195_060HPI
#C195_60H_2	060HPI	C195	R2	Susceptible	hypocotyl	C195_060HPI
sample_metadata = read.delim("your_path/sampledatainfo.txt",
                      header=T)

In [ ]:
#Create directories

path<-"your_path/WGCNA"
dir.create(paste0(path,"/plots"),recursive = FALSE,showWarnings = FALSE)
plots<-paste0(path,"/plots")

dir.create(paste0(path,"/tables"),recursive = FALSE,showWarnings = FALSE)
tables<-paste0(path,"/tables")

setwd(path)

In [ ]:
#Delete unecessary column
expr_matrix <- tmm_norm[,-1]
rownames(expr_matrix) <- tmm_norm[,1]

In [ ]:
#Columns of expr_matrix must be the same name and order than rows in sample_metadata
colnames(expr_matrix)
sample_metadata$sample  
all(colnames(expr_matrix) == sample_metadata$sample)

In [ ]:
#Normalize TMM counts with log2

cuentasnormalizadas <- log2(expr_matrix + 1)
write.table(cuentasnormalizadas, "normCounts_log2_tmm.tsv", sep = "\t", quote = FALSE, row.names = TRUE)

In [ ]:
#Now samples in rows and genes in columns
datExpr <- t(log2(expr_matrix + 1))


In [29]:
# Calculate sample distance and cluster the samples, OUTLIERS detection
sampleTree = hclust(dist(datExpr), method = "average");

In [ ]:
#plot sampleTree
plot(sampleTree, main = "Sample clustering to detect outliers", sub="", xlab="",
     cex.lab = 1.5,cex.axis = 1.5, cex.main = 2)

In [31]:
#save sampleTree clustering 
pdf(file = paste0(plots,"/1_sampleClustering.pdf"), width = 40, height = 9);
par(cex = 1.3);
par(mar = c(0,4,2,0))
plot(sampleTree, main = "Sample clustering to detect outliers", sub="", xlab="",
     cex.lab = 1.5,cex.axis = 1.5, cex.main = 2)
dev.off()


agg_record_1343482132 
                    2

In [ ]:
#Choose a set of soft threshold parameters
powers = c(c(1:10), seq(from = 12, to=30, by=2))


In [ ]:
sft = pickSoftThreshold(datExpr, powerVector = powers, verbose = 5, networkType = "unsigned") 

In [34]:
#save soft threshold table
write.table(sft$fitIndices,file = paste0(tables,"/1_sft_threshold_indices_unsigned.tsv"),sep = '\t',quote = F)

In [ ]:
par(mfrow = c(1,2));
cex1 = .8;
plot(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     xlab="Soft Threshold (power)",ylab="Scale Free Topology Model Fit,signed R^2",type="n",
     main = paste("Scale independence"));
text(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     labels=powers,cex=cex1,col="red");
# this line corresponds to using an R^2 cut-off of h
abline(h=0.9,col="red") 



In [ ]:
# Mean connectivity as a function of the soft-thresholding power
plot(sft$fitIndices[,1], sft$fitIndices[,5],
     xlab="Soft Threshold (power)",ylab="Mean Connectivity", type="n",
     main = paste("Mean connectivity")) 
text(sft$fitIndices[,1], sft$fitIndices[,5], labels=powers, cex=cex1,col="red")


In [37]:
# Scale-free topology fit index as a function of the soft-thresholding power
pdf(file = paste0(plots,"/2_n_sft.pdf"), width = 9, height = 5);
par(mfrow = c(1,2));
cex1 = 0.9;
plot(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     xlab="Soft Threshold (power)",ylab="Scale Free Topology Model Fit,signed R^2",type="n",
     main = paste("Scale independence"));
text(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     labels=powers,cex=cex1,col="red");
# this line corresponds to using an R^2 cut-off of h
abline(h=0.90,col="red") 
# Mean connectivity as a function of the soft-thresholding power
plot(sft$fitIndices[,1], sft$fitIndices[,5],
     xlab="Soft Threshold (power)",ylab="Mean Connectivity", type="n",
     main = paste("Mean connectivity")) 
text(sft$fitIndices[,1], sft$fitIndices[,5], labels=powers, cex=cex1,col="red")
dev.off()

agg_record_1376035638 
                    2

In [ ]:
power=12

In [ ]:
enableWGCNAThreads(nThreads = 15)

Allowing parallel execution with up to 15 working processes.


In [ ]:

net <- blockwiseModules(datExpr,
                       power = power,
                       networkType = "unsigned",   # or "signed"
                       TOMType = "unsigned",       # debe concordar con networkType
                       minModuleSize = 30,  
                       reassignThreshold = 0,
                       mergeCutHeight = 0.25, 
                       numericLabels = TRUE, #choose FALSE if you preffer colors labels
                       pamRespectsDendro = FALSE,
                       verbose = 3,
                       nThreads = 15)



In [ ]:
#Check if all genes were analyzed
length(net$colors)  #genes
length(unique(net$colors)) #modules

[1] 33255

[1] 22

In [ ]:
#check genes by modules
table(net$colors)


   0    1    2    3    4    5    6    7    8    9   10   11   12   13   14   15 
3074 6195 5617 3368 3058 2929 2421 1358 1350  824  674  523  427  400  313  207 
  16   17   18   19   20   21 
 162  127   77   57   56   38 

In [ ]:
#dendrograma
pdf("3_cluster_dendrogram.pdf", width = 10, height = 7)
plotDendroAndColors(net$dendrograms[[1]],
                    net$colors[net$blockGenes[[1]]],
                    "Module colors",
                    dendroLabels = FALSE,
                    hang = 0.03,
                    addGuide = TRUE,
                    guideHang = 0.05)
dev.off()


In [45]:
save.image(file = "WGCNA_Part1.RData")

In [ ]:

enableWGCNAThreads()
# TOM calculation
TOM <- TOMsimilarityFromExpr(datExpr, power = power, TOMType = "unsigned")

In [ ]:
genes <- colnames(datExpr)
colors <- net$colors 


In [ ]:
#Heatmap1
mergedMEs = net$MEs


pdf(file=paste0(plots,"/4_modules_vs_samples_nocategory_heatmap.pdf"),height=10,width=10)
rownames(mergedMEs) = rownames(datExpr)
scaledMEs = t(scale(t(mergedMEs)))

pheatmap(scaledMEs, 
         cluster_rows = TRUE,
         cluster_cols = TRUE,
         show_colnames = TRUE,
         show_rownames = TRUE,
         fontsize_row = 8,
         fontsize_col = 6,
         main = "Heatmap modules by treatment non categorized",
         filename = paste0(plots, "/4_modules_vs_samples_nocategory_heatmap.pdf"),
         width = 10,
         height = 10)


In [ ]:
# Heatmap2
col_ann <- sample_metadata[,c(1,7)] 
#Check, rows in sample_metadata must be in the same order in the datEXpr object
all(rownames(datExpr) == sample_metadata[,1])

rownames(col_ann) <- col_ann[,1] 
col_ann <- data.frame(col_ann) 
col_ann$treatment <- as.factor(col_ann$treatment) 
col_ann <- col_ann[order(col_ann$treatment),]
col_ann$sample <- NULL

In [28]:
ann_color <- list("col_ann" = c("C195_000HPI" = "#f0f921",
                                "C195_024HPI" = "#febd2a",
                                "C195_060HPI" = "#f48849",
                                "C195_084HPI" = "#db5c68", 
                                "C98_000HPI" = "#b83289", 
                                "C98_024HPI" = "#8b0aa5", 
                                "C98_060HPI" = "#5302a3",
                                "C98_084HPI" = "#0d0887"
                               ))

In [30]:
data <- data.frame(mergedMEs)
data <- data[order(match(rownames(data), rownames(col_ann))),]

In [ ]:
pdf(file=paste0(plots,"/5_modules_vs_treatment_category_heatmap.pdf"),height=10,width=10)

pheatmap(data,
         color = hcl.colors(50, "RdYlBu",rev = T),
         cluster_col=T,
         cluster_row=F,
         show_rownames=F,
         show_colnames=T,
         fontsize=6,
         annotation_row = col_ann, 
         annotation_colors = ann_color,
         main = "Heatmap modules by treatment categorized",
         filename = paste0(plots, "/5_modules_vs_treatment_category_heatmap.pdf"),
         width = 10,
         height = 10)

In [ ]:
#Binary file of traits, where 1 when sample (row) corresponds to the traitment (columns)
#                       where 0 when sample (row) no corresponds to the traitment (columns)
#file looks like:
#X	Resistant	Susceptible	S_000hpi	S_024hpi	S_060hpi	S_084hpi	R_000hpi	R_024hpi	R_060hpi	R_084hpi
#C98_C1	1	0	0	0	0	0	1	0	0	0
#C98_C2	1	0	0	0	0	0	1	0	0	0

traits<-read.delim("your_path/binary_traits.tsv",header = TRUE,row.names = 1)

In [ ]:
# numbers of genes and samples
nGenes = ncol(datExpr); 
nSamples = nrow(datExpr);

In [ ]:
# Recalculate MEs with color labels
MEs0 = moduleEigengenes(datExpr, mergedColors)$eigengenes 
MEs = orderMEs(MEs0) 

In [10]:
# Calculate pearson correlation coefficients between modules and traits, p -values
moduleTraitCor = cor(MEs, traits, use = "p");
moduleTraitPvalue = corPvalueStudent(moduleTraitCor, nSamples);

In [ ]:
#re order modules
modules_names <- colnames(MEs)

modules_num <- as.numeric(sub("ME", "", modules_names))

order_modules <- order(modules_num)

modules_sorted <- modules_names[order_modules]


In [ ]:
#heatmap colors
my.colors <- colorRampPalette(c("#a6dba0","#F0FFF0", "#BF3EFF"))(50) 

In [ ]:
# Important plot heatmap of module-traits relationship
sizeGrWindow(10,10)

# Will display correlations and their p-values
textMatrix =  paste(signif(moduleTraitCor, 2), "\n(",
                    signif(moduleTraitPvalue, 1), ")", sep = "");
dim(textMatrix) = dim(moduleTraitCor)
pdf("your_path/module_traits.pdf", width = 20, height = 20)
par(mar = c(15, 12, 5, 5));

# Display the correlation values within a heatmap plot
labeledHeatmap(Matrix = moduleTraitCor[order_modules, ],
               xLabels = colnames(traits),
               yLabels = modules_sorted,
               ySymbols = modules_sorted,
               colorLabels = FALSE,
               colors = my.colors,
               textMatrix = textMatrix[order_modules, ], 
               setStdMargins = FALSE,
               cex.text = 1.7,
               zlim = c(-1,1),
               cex.lab.x    = 1.7,
               cex.lab.y    = 1.7,
               xLabelsAngle = 45, 
               xColorOffset  = strheight("M") * 1.4,
               yColorOffset  = strwidth("M") * 1.4,
               plotLegend= TRUE,    
               main = NULL)
dev.off()



png 
  2

In [79]:
save.image(file = "WGCNA.RData")

In [ ]:
#save dataframe heatmap Modules, traits and p value

heatmap.data <- merge(MEs, traits, by = 'row.names')

head(heatmap.data)


In [81]:
resdata <- merge(as.data.frame(moduleTraitCor), 
                 as.data.frame(moduleTraitPvalue), 
                 by = 'row.names', sort = FALSE)

In [ ]:
tables <- "your/path"

In [86]:
write.table(resdata,file = paste0(tables,"/ModuleTraitCorrelation.tsv"),
            sep = "\t", row.names = F)

In [ ]:
modNames = substring(names(MEs), 3) 
geneModuleMembership = as.data.frame(cor(datExpr, MEs, use = "p"))
MMPvalue = as.data.frame(corPvalueStudent(as.matrix(geneModuleMembership), nSamples))
names(geneModuleMembership) = paste("MM", modNames, sep="");
names(MMPvalue) = paste("p.MM", modNames, sep="");

In [ ]:
#Key step: Save relationships between genes and co-expressed modules
write.table(x = geneModuleMembership,file = paste0(tables,"/geneModuleMembership.csv"), sep = '\t')

In [ ]:
#Key step: Save p-values of relationships between genes and co-expressed modules
write.table(x = MMPvalue,file = paste0(tables,"/PvaluegeneModuleMembership.csv"), sep = '\t')

In [ ]:
#Key step: BIGNET GENERATION, which include all relationship between total genes! 
colors <- net$colors
genes <- colnames(datExpr)
modProbes <- genes 
modGenes  <- genes 
modTOM    <- TOM
dimnames(modTOM) <- list(modProbes, modProbes)
cyt = exportNetworkToCytoscape(modTOM,
                               edgeFile = "your_path/bigNet_edges.txt",
                               nodeFile =  "your_path/bigNet_nodes.txt",
                               weighted = TRUE,
                               threshold = 0.02,
                               nodeNames = modProbes,
                               altNodeNames = modGenes,
                               nodeAttr = colors)

In [ ]:
#GOOD! Following, we calculate de relationships between genes and treatments  


#RESISTANT traits

resistant=as.data.frame(traits$Resistant)
names(resistant) = "Resistant"

r00hpi= as.data.frame(traits$R_000hpi)
names(r00hpi) = "R_000hpi"
r24hpi= as.data.frame(traits$R_024hpi)
names(r24hpi) = "R_024hpi"
r60hpi= as.data.frame(traits$R_060hpi)
names(r60hpi) = "R_060hpi"
r84hpi= as.data.frame(traits$R_084hpi)
names(r84hpi) = "R_084hpi"

In [ ]:
#Resistant trait: genes related to all resistant conditions

geneTraitSignificance_resistant = as.data.frame(cor(datExpr,resistant, use = "p")); 
GSPvalue_resistant= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_resistant), nSamples)); 
names(geneTraitSignificance_resistant) = paste("GS.", names(resistant), sep="");
names(GSPvalue_resistant) = paste("p.GS.", names(resistant), sep="");


geneTraitSignificance_resistant$GS_abs_resistant = abs(geneTraitSignificance_resistant[[1]])

write.table(x = geneTraitSignificance_resistant,file = paste0(tables,"/geneTraitSignificance_resistant.csv"), sep = '\t')

write.table(x = GSPvalue_resistant,file = paste0(tables,"/GeneSignificancePvalue_resistant.csv"), sep = '\t')


In [ ]:
#Resistant trait: Genes related to 0 hpi (uninfected)
geneTraitSignificance_r00hpi = as.data.frame(cor(datExpr,r00hpi, use = "p"));
GSPvalue_r00hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_r00hpi), nSamples));
names(geneTraitSignificance_r00hpi) = paste("GS.", names(r00hpi), sep="");
names(GSPvalue_r00hpi) = paste("p.GS.", names(r00hpi), sep="");

geneTraitSignificance_r00hpi$GS_abs_r00hpi = abs(geneTraitSignificance_r00hpi[[1]])

write.table(x = geneTraitSignificance_r00hpi,file = paste0(tables,"/geneTraitSignificance_r00hpi.csv"), sep = '\t')

write.table(x = GSPvalue_r00hpi,file = paste0(tables,"/GeneSignificancePvalue_r00hpi.csv"), sep = '\t')

In [ ]:
#Resistant trait: Genes related to 24 hpi 
geneTraitSignificance_r24hpi = as.data.frame(cor(datExpr,r24hpi, use = "p"));
GSPvalue_r24hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_r24hpi), nSamples));
names(geneTraitSignificance_r24hpi) = paste("GS.", names(r24hpi), sep="");
names(GSPvalue_r24hpi) = paste("p.GS.", names(r24hpi), sep="");

geneTraitSignificance_r24hpi$GS_abs_r24hpi = abs(geneTraitSignificance_r24hpi[[1]])

write.table(x = geneTraitSignificance_r24hpi,file = paste0(tables,"/geneTraitSignificance_r24hpi.csv"), sep = '\t')

write.table(x = GSPvalue_r24hpi,file = paste0(tables,"/GeneSignificancePvalue_r24hpi.csv"), sep = '\t')

In [ ]:
#Resistant trait: Genes related to 60 hpi 

geneTraitSignificance_r60hpi = as.data.frame(cor(datExpr,r60hpi, use = "p"));
GSPvalue_r60hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_r60hpi), nSamples));
names(geneTraitSignificance_r60hpi) = paste("GS.", names(r60hpi), sep="");
names(GSPvalue_r60hpi) = paste("p.GS.", names(r60hpi), sep="");


geneTraitSignificance_r60hpi$GS_abs_r60hpi = abs(geneTraitSignificance_r60hpi[[1]])

write.table(x = geneTraitSignificance_r60hpi,file = paste0(tables,"/geneTraitSignificance_r60hpi.csv"), sep = '\t')

write.table(x = GSPvalue_r60hpi,file = paste0(tables,"/GeneSignificancePvalue_r60hpi.csv"), sep = '\t')

In [ ]:
#Resistant trait: Genes related to 84 hpi 

geneTraitSignificance_r84hpi = as.data.frame(cor(datExpr,r84hpi, use = "p"));
GSPvalue_r84hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_r84hpi), nSamples));
names(geneTraitSignificance_r84hpi) = paste("GS.", names(r84hpi), sep="");
names(GSPvalue_r84hpi) = paste("p.GS.", names(r84hpi), sep="");

geneTraitSignificance_r84hpi$GS_abs_r84hpi = abs(geneTraitSignificance_r84hpi[[1]])

write.table(x = geneTraitSignificance_r84hpi,file = paste0(tables,"/geneTraitSignificance_r84hpi.csv"), sep = '\t')

write.table(x = GSPvalue_r84hpi,file = paste0(tables,"/GeneSignificancePvalue_r84hpi.csv"), sep = '\t')

In [ ]:
#SUSCEPTIBLE traits

susceptible=as.data.frame(traits$Susceptible)
names(susceptible) = "Susceptible"

s00hpi= as.data.frame(traits$S_000hpi)
names(s00hpi) = "S_000hpi"
s24hpi= as.data.frame(traits$S_024hpi)
names(s24hpi) = "S_024hpi"
s60hpi= as.data.frame(traits$S_060hpi)
names(s60hpi) = "S_060hpi"
s84hpi= as.data.frame(traits$S_084hpi)
names(s84hpi) = "S_084hpi"

In [ ]:
#Susceptible trait: genes related to all susceptible conditions

geneTraitSignificance_susceptible = as.data.frame(cor(datExpr,susceptible, use = "p")); # la correlación (Pearson) entre cada gen y el trait 
GSPvalue_susceptible= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_susceptible), nSamples)); # el valor p asociado a esa correlación.
names(geneTraitSignificance_susceptible) = paste("GS.", names(susceptible), sep="");
names(GSPvalue_susceptible) = paste("p.GS.", names(susceptible), sep="");

geneTraitSignificance_susceptible$GS_abs_susceptible = abs(geneTraitSignificance_susceptible[[1]])

write.table(x = geneTraitSignificance_susceptible,file = paste0(tables,"/geneTraitSignificance_susceptible.csv"), sep = '\t')

write.table(x = GSPvalue_susceptible,file = paste0(tables,"/GeneSignificancePvalue_susceptible.csv"), sep = '\t')

In [ ]:
#Susceptible trait: Genes related to 0 hpi (uninfected)
geneTraitSignificance_s00hpi = as.data.frame(cor(datExpr,s00hpi, use = "p"));
GSPvalue_s00hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_s00hpi), nSamples));
names(geneTraitSignificance_s00hpi) = paste("GS.", names(s00hpi), sep="");
names(GSPvalue_s00hpi) = paste("p.GS.", names(s00hpi), sep="");

geneTraitSignificance_s00hpi$GS_abs_s00hpi = abs(geneTraitSignificance_s00hpi[[1]])

write.table(x = geneTraitSignificance_s00hpi,file = paste0(tables,"/geneTraitSignificance_s00hpi.csv"), sep = '\t')

write.table(x = GSPvalue_s00hpi,file = paste0(tables,"/GeneSignificancePvalue_s00hpi.csv"), sep = '\t')

In [ ]:
#Susceptible trait: Genes related to 24 hpi 
geneTraitSignificance_s24hpi = as.data.frame(cor(datExpr,s24hpi, use = "p"));
GSPvalue_s24hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_s24hpi), nSamples));
names(geneTraitSignificance_s24hpi) = paste("GS.", names(s24hpi), sep="");
names(GSPvalue_s24hpi) = paste("p.GS.", names(s24hpi), sep="");

geneTraitSignificance_s24hpi$GS_abs_s24hpi = abs(geneTraitSignificance_s24hpi[[1]])

write.table(x = geneTraitSignificance_s24hpi,file = paste0(tables,"/geneTraitSignificance_s24hpi.csv"), sep = '\t')

write.table(x = GSPvalue_s24hpi,file = paste0(tables,"/GeneSignificancePvalue_s24hpi.csv"), sep = '\t')

In [ ]:
#Susceptible trait: Genes related to 60 hpi 
geneTraitSignificance_s60hpi = as.data.frame(cor(datExpr,s60hpi, use = "p"));
GSPvalue_s60hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_s60hpi), nSamples));
names(geneTraitSignificance_s60hpi) = paste("GS.", names(s60hpi), sep="");
names(GSPvalue_s60hpi) = paste("p.GS.", names(s60hpi), sep="");

geneTraitSignificance_s60hpi$GS_abs_s60hpi = abs(geneTraitSignificance_s60hpi[[1]])

write.table(x = geneTraitSignificance_s60hpi,file = paste0(tables,"/geneTraitSignificance_s60hpi.csv"), sep = '\t')

write.table(x = GSPvalue_s60hpi,file = paste0(tables,"/GeneSignificancePvalue_s60hpi.csv"), sep = '\t')

In [ ]:
#Susceptible trait: Genes related to 84 hpi 
geneTraitSignificance_s84hpi = as.data.frame(cor(datExpr,s84hpi, use = "p"));
GSPvalue_s84hpi= as.data.frame(corPvalueStudent(as.matrix(geneTraitSignificance_s84hpi), nSamples));
names(geneTraitSignificance_s84hpi) = paste("GS.", names(s84hpi), sep="");
names(GSPvalue_s84hpi) = paste("p.GS.", names(s84hpi), sep="");

geneTraitSignificance_s84hpi$GS_abs_s84hpi = abs(geneTraitSignificance_s84hpi[[1]])

write.table(x = geneTraitSignificance_s84hpi,file = paste0(tables,"/geneTraitSignificance_s84hpi.csv"), sep = '\t')

write.table(x = GSPvalue_s84hpi,file = paste0(tables,"/GeneSignificancePvalue_s84hpi.csv"), sep = '\t')

In [ ]:
#GOOD JOB :) 
#WGCNA analysis is complete! --------------------------------------------------------------------- 


In [ ]:
#Final plots 

MET = orderMEs(cbind(MEs, resistant))

In [ ]:
# Plot the dendrogram
sizeGrWindow(6,6);
pdf(paste0(plots,"/7_module_resistant.pdf"), width = 20, height = 15,)
par(cex = 1.0)
plotEigengeneNetworks(MET, "Eigengene dendrogram", marDendro = c(0,4,2,0),
                      plotHeatmaps = FALSE)
# Plot the heatmap matrix (note: this plot will overwrite the dendrogram plot)
par(cex = 1.0)
plotEigengeneNetworks(MET, "Eigengene adjacency heatmap", marHeatmap = c(10,10,1,2),
                      plotDendrograms = FALSE, xLabelsAngle = 90)
dev.off()

png 
  2

In [ ]:
#plots of specific modules
module = "2"
moduleColors = mergedColors
column = match(module, modNames)
moduleGenes = moduleColors == module

GS_abs_resistant = geneTraitSignificance_resistant$GS_abs_resistant

sizeGrWindow(7, 7)
pdf(paste0(plots,"/8_module_resistant_", module, ".pdf"), width = 20, height = 15)
par(mfrow = c(1,1))

verboseScatterplot(
  abs(geneModuleMembership[moduleGenes, column]),
  GS_abs_resistant[moduleGenes],
  xlab = paste("Module Membership", module, "2"),
  ylab = "Gene significance (abs) for resistance",
  main = paste("Module membership vs. gene significance\n", module),
  cex.main = 1.2, cex.lab = 1.2, cex.axis = 1.2,
  col = module
)
dev.off()

png 
  2

In [ ]:
#plots of specific modules
module = "5"
moduleColors = mergedColors
column = match(module, modNames)
moduleGenes = moduleColors == module

GS_abs_resistant = geneTraitSignificance_resistant$GS_abs_resistant

sizeGrWindow(7, 7)
pdf(paste0(plots,"/9_module_resistant_", module, ".pdf"), width = 20, height = 15)
par(mfrow = c(1,1))

verboseScatterplot(
  abs(geneModuleMembership[moduleGenes, column]),
  GS_abs_resistant[moduleGenes],
  xlab = paste("Module Membership", module, "5"),
  ylab = "Gene significance (abs) for resistance",
  main = paste("Module membership vs. gene significance\n", module),
  cex.main = 1.2, cex.lab = 1.2, cex.axis = 1.2,
  col = module
)
dev.off()

png 
  2

In [ ]:
#plots of specific modules
module = "11"
moduleColors = mergedColors
column = match(module, modNames)
moduleGenes = moduleColors == module

# Obtén tu vector custom de GS absoluta
GS_abs_resistant = geneTraitSignificance_resistant$GS_abs_resistant

sizeGrWindow(7, 7)
pdf(paste0(plots,"/10_module_resistant_", module, ".pdf"), width = 20, height = 15)
par(mfrow = c(1,1))

verboseScatterplot(
  abs(geneModuleMembership[moduleGenes, column]),
  GS_abs_resistant[moduleGenes],
  xlab = paste("Module Membership", module, "11"),
  ylab = "Gene significance (abs) for resistance",
  main = paste("Module membership vs. gene significance\n", module),
  cex.main = 1.2, cex.lab = 1.2, cex.axis = 1.2,
  col = module
)
dev.off()

png 
  2

In [ ]:
#SAVE and good luck!
save.image(file = "WGCNA.RData")